# Baseline — Coin Value Counting

**Competition:** given a photo of Polish coins, predict the **total value in grosz**
(1 złoty = 100 grosz).

- **Task:** regression on images
- **Metric:** error in grosz (lower is better)
- **Kaggle link:** _TODO: add link_

**Approach:** frozen ResNet-18 features + Ridge regression as a quick global baseline.
(The intended lesson is classic computer vision: detect circles, classify each coin,
sum the values — see ideas below.)

In [1]:
import numpy as np
import pandas as pd
import torch
from torchvision.models import resnet18, ResNet18_Weights
from PIL import Image

DATA_DIR = "."
train = pd.read_csv(f"{DATA_DIR}/train.csv")
test  = pd.read_csv(f"{DATA_DIR}/test.csv")
print(train.shape, test.shape)
train.head(3)

(49, 3) (21, 2)


,id,image_path,total_value_grosz
0,train_00000,train_images/train_00000.png,883
1,train_00001,train_images/train_00001.png,992
2,train_00002,train_images/train_00002.png,1652


In [2]:
device = "cuda" if torch.cuda.is_available() else "cpu"
weights = ResNet18_Weights.IMAGENET1K_V1
backbone = resnet18(weights=weights); backbone.fc = torch.nn.Identity()
backbone.eval().to(device)
preprocess = weights.transforms()

@torch.no_grad()
def extract(paths, bs=16):
    out = []
    for i in range(0, len(paths), bs):
        b = [preprocess(Image.open(f"{DATA_DIR}/{p}").convert("RGB")) for p in paths[i:i+bs]]
        out.append(backbone(torch.stack(b).to(device)).cpu().numpy())
    return np.vstack(out)

X  = extract(train["image_path"].tolist())
Xt = extract(test["image_path"].tolist())
print(X.shape, Xt.shape)

(49, 512) (21, 512)


In [3]:
from sklearn.linear_model import Ridge
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import mean_absolute_error

y = train["total_value_grosz"].values
reg = Ridge(alpha=10.0)
oof = np.clip(cross_val_predict(reg, X, y, cv=5), 0, None)
print(f"CV MAE: {mean_absolute_error(y, oof):.1f} grosz  (mean target {y.mean():.0f})")

CV MAE: 321.9 grosz  (mean target 711)


In [4]:
reg.fit(X, y)
pred = np.clip(reg.predict(Xt), 0, None)
sub = pd.DataFrame({"id": test["id"], "total_value_grosz": pred})
sub.to_csv("submission.csv", index=False)
sub.head()

,id,total_value_grosz
0,test_00000,529.947998
1,test_00001,1132.240479
2,test_00002,694.420776
3,test_00003,280.543243
4,test_00004,601.062561


## Ideas to improve

- The real approach: **Hough circle detection** (OpenCV `HoughCircles`) to find each
  coin, then classify each crop by relative size + color into a denomination, and sum
  `coin_values.csv` values — this is how the original "Maszynka do Liczenia Monet"
  notebook works.
- Or modern: train a small object detector (YOLO) on coin crops.
- Fine-tune the CNN as a *counting* regressor with strong augmentation on GPU.
